In [54]:
import torch
import numpy as np
from botorch import fit_gpytorch_model
from botorch.optim.optimize import optimize_acqf
from botorch.acquisition.multi_objective.monte_carlo import qExpectedHypervolumeImprovement
from botorch.models.gp_regression import SingleTaskGP
from botorch.models.transforms.outcome import Standardize
from gpytorch.mlls.exact_marginal_log_likelihood import ExactMarginalLogLikelihood
from botorch.utils.transforms import unnormalize
from botorch.utils.multi_objective.box_decompositions.non_dominated import NondominatedPartitioning
from botorch.utils.sampling import draw_sobol_samples
from botorch.utils.multi_objective.hypervolume import Hypervolume
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.sampling.samplers import SobolQMCNormalSampler
torch.manual_seed(42)

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_objs    = 2
num_params  = 4                                                     # Number of design parameters
batch_size  = 1                                                     # Number of design parameter points to query at next iteration
num_restart = 10                                                    # Used for the acquisition function number of restarts in optimization
raw_samples = 1024                                                  # Initial restart location candidates
ref_point   = torch.tensor([-1. for _ in range(num_objs)]).cuda()   # Reference point in objective function space
mc_samples  = 512 # Number of samples to approximate acquisition function

n_initial = 5
max_iterations = 20

# Parameter space
problem_bounds = torch.stack([torch.zeros(num_params), torch.ones(num_params)]).to(device, dtype=torch.double)

# Use Sobol samples as it tends to produce more accurate (i.e. lower variance)
# gradient estimates with much fewer samples relative to the IIDNormalSampler.
# See https://botorch.org/docs/samplers#docsNav
train_x = draw_sobol_samples(bounds=problem_bounds, n=1, q=6).squeeze(0)
train_x

tensor([[0.9975, 0.1044, 0.8230, 0.4194],
        [0.5283, 0.0562, 0.4553, 0.7492],
        [0.2211, 0.6177, 0.8101, 0.2123],
        [0.0384, 0.8818, 0.6506, 0.6666],
        [0.8209, 0.5966, 0.2363, 0.8070],
        [0.1671, 0.7678, 0.6035, 0.9326]], device='cuda:0',
       dtype=torch.float64)

In [91]:
def objective_function(x: torch.Tensor):
    # TODO:
    # 1. present design based on parameter x to human
    # 2. collect response from human as objectives
    x = x.cpu().numpy()
    return torch.tensor([1, 1], dtype=torch.float64).to(device)

objective_function(train_x)

tensor([1., 1.], device='cuda:0', dtype=torch.float64)

In [92]:
# Initial Samples
train_x_qehvi = draw_sobol_samples(bounds=problem_bounds, n=1, q=n_initial).squeeze(0)
train_obj = []
for i, x in enumerate(train_x_qehvi):
    train_obj.append(objective_function(x)) # expensive operation

train_y_qehvi = torch.tensor([item.cpu().detach().numpy() for item in train_obj], dtype=torch.float64).to(device)
train_y_qehvi

tensor([[1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.]], device='cuda:0', dtype=torch.float64)

In [102]:
import matplotlib.pyplot as plt
import pandas as pd
import time

start_time = time.strftime("%Y-%m-%d-%H-%M", time.localtime())
def save_xy(x_sample, y_sample, hvs_qehvi):
    pareto_mask = is_non_dominated(y_sample)
    pareto_obj = y_sample[pareto_mask]

    x_sample = x_sample.cpu().numpy()
    y_sample = y_sample.cpu().numpy()
    pareto_obj = pareto_obj.cpu().numpy()
    pareto_front = x_sample[pareto_mask.cpu()]
    all_record = np.concatenate((y_sample,x_sample), axis = 1)

    f_values = y_sample.copy()
    f_values = np.array([ list(x) for x in f_values])

    x_all = f_values[:, 0]
    y_all = f_values[:, 1]
    pareto_obj = pareto_obj[pareto_obj[:, 0].argsort()]
    x_pareto = pareto_obj[:, 0]
    y_pareto = pareto_obj[:, 1]

    # Create parallel coordinates plot

    line_index = list(range(len(pareto_front)))
    pareto_design_parameters = np.concatenate((np.array([line_index]).T, pareto_front), axis = 1)

    columns_i = ["iter"]
    for i in range(len(pareto_design_parameters[0])-1):
        columns_i.append("x" + str(i+1))

    design_parameters_pd = pd.DataFrame(data=pareto_design_parameters, index=line_index, columns=columns_i)

    plt.figure(figsize=(15, 6))
    plt.subplot(121)
    plt.title('Objective values')
    plt.scatter(x_all, y_all)
    plt.plot(x_pareto, y_pareto, color='r')
    plt.xlabel('Completion Time')
    plt.ylabel('Accuracy')
    plt.xlim(-1, 1)
    plt.ylim(-1, 1)

    plt.subplot(122)
    plt.title('Design parameters')
    pd.plotting.parallel_coordinates(design_parameters_pd, "iter")
    plt.savefig('assets/observations', dpi=50)
    plt.clf()
    plt.figure()
    plt.plot(hvs_qehvi)
    plt.title("Pareto Hypervolume Increase", fontsize=24)
    plt.tick_params(axis='x', labelsize=16)
    plt.tick_params(axis='y', labelsize=16)
    plt.savefig('assets/hypervolume', dpi=50)
    plt.clf()

    # add new column to identify pareto points
    index_arr = []
    for i in pareto_mask:
        temp = ""
        if(i):
            temp = "TRUE"
        else:
            temp = "FALSE"
        index_arr.append(temp)
    all_record = np.concatenate((all_record, np.array([index_arr]).T), axis=1)
    header = np.array(['CompletionTime', 'SpatialError', 'D', 'K', 'Amplitude', 'Gap', 'IsPareto'])
    all_record = np.append([header],all_record,axis=0)
    np.savetxt('{}ObservationsPerEvalution.csv'.format(start_time), all_record, delimiter=',', fmt="%s")
    np.savetxt('UserObservations.csv', all_record, delimiter=',', fmt="%s")
    np.savetxt('{}HypervolumePerEvalution.csv'.format(start_time), np.array(hvs_qehvi))

In [103]:
# Initialize GP models, define models for objective and constraint
model_qehvi = SingleTaskGP(train_x_qehvi, train_y_qehvi, outcome_transform=Standardize(m=train_y_qehvi.shape[-1]))
mll_qehvi   = ExactMarginalLogLikelihood(model_qehvi.likelihood, model_qehvi)
hvs_qehvi   = []
hv          = Hypervolume(ref_point=ref_point)

# Compute Pareto front and hypervolume
volume = hv.compute(train_y_qehvi[is_non_dominated(train_y_qehvi)])
hvs_qehvi.append(volume)

save_xy(train_x_qehvi, train_y_qehvi, hvs_qehvi)

<Figure size 1080x432 with 0 Axes>

<Figure size 432x288 with 0 Axes>

In [97]:
def optimize_qehvi(model, train_obj):
    '''Optimizes the qEHVI acquisition function, and returns a new candidate and observation.'''
    candidates, _ = optimize_acqf(
        acq_function=qExpectedHypervolumeImprovement(
            model=model,
            ref_point=ref_point.tolist(), # use known reference point
            # partition non-dominated space into disjoint rectangles
            partitioning=NondominatedPartitioning(ref_point=ref_point, Y=train_obj),
            sampler=SobolQMCNormalSampler(num_samples=mc_samples),
        ),
        bounds=problem_bounds,
        q=batch_size,
        num_restarts=num_restart,
        raw_samples=raw_samples, # used for intialization heuristic
        options={'batch_limit': 5, 'maxiter': 200, 'nonnegative': True},
        sequential=True,
    )

    # observe new values
    return unnormalize(candidates.detach(), bounds=problem_bounds)

In [104]:
for iteration in range(1, max_iterations + 1):
    print('Iteration: ' + str(iteration))
    # Fit Models
    fit_gpytorch_model(mll_qehvi)

    # Optimize acquisition functions and get new observations
    new_x_qehvi = optimize_qehvi(model_qehvi, train_y_qehvi)
    new_y_qehvi = objective_function(new_x_qehvi[0]) # expensive operation

    # Update training points
    train_x_qehvi = torch.cat([train_x_qehvi, new_x_qehvi])
    train_y_qehvi = torch.cat([train_y_qehvi, new_y_qehvi.unsqueeze(0)])

    # Compute hypervolumes
    volume = hv.compute(train_y_qehvi[is_non_dominated(train_y_qehvi)])
    hvs_qehvi.append(volume)

    save_xy(train_x_qehvi, train_y_qehvi, hvs_qehvi)

    model_qehvi = SingleTaskGP(train_x_qehvi, train_y_qehvi, outcome_transform=Standardize(m=train_y_qehvi.shape[-1]))
    mll_qehvi   = ExactMarginalLogLikelihood(model_qehvi.likelihood, model_qehvi)

print(train_x_qehvi, train_y_qehvi)

Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
Iteration: 9
Iteration: 10
Iteration: 11


/tmp/ipykernel_661463/4149125772.py:36: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  plt.figure(figsize=(15, 6))


Iteration: 12
Iteration: 13
Iteration: 14
Iteration: 15
Iteration: 16
Iteration: 17
Iteration: 18
Iteration: 19
Iteration: 20
tensor([[0.5835, 0.8163, 0.3899, 0.7428],
        [0.7537, 0.4308, 0.2683, 0.3693],
        [0.6026, 0.0581, 0.2255, 0.7133],
        [0.1075, 0.9471, 0.4795, 0.0177],
        [0.0102, 0.1807, 0.4324, 0.1078],
        [0.0000, 0.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 0.0000],
        [1.0000, 0.0000, 1.0000, 1.0000],
        [1.0000, 0.0000, 1.0000, 0.0000],
        [0.0000, 1.0000, 1.0000, 1.0000],
        [0.0000, 0.9724, 0.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 1.0000, 0.0000, 1.0000],
        [0.3205, 0.4605, 1.0000, 0.3270],
        [0.0000, 1.0000, 0.0000, 0.2806],
        [1.0000, 0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 1.0000, 0.0000],
        [0.0000, 1.0000, 1.0000, 0

<Figure size 1080x432 with 0 Axes>

<Figure size 432x288 with 0 Axes>

...

<Figure size 432x288 with 0 Axes>

<Figure size 1080x432 with 0 Axes>

<Figure size 432x288 with 0 Axes>

<Figure size 1080x432 with 0 Axes>

<Figure size 432x288 with 0 Axes>